In [ ]:
# Data root is configurable: export SYNERGPCR_BASE=/path/to/released/tables
# (defaults to ./data). All input paths below are resolved against it.
from pathlib import Path
import os
import pandas as pd
import time
import json
import re
from tqdm import tqdm
import google.generativeai as genai
from typing_extensions import TypedDict

BASE = Path(os.environ.get("SYNERGPCR_BASE", "./data"))


# ==========================================
# 1. API Configuration
# ==========================================
API_KEY = os.environ.get("GEMINI_API_KEY")  # Insert your API key here
genai.configure(api_key=API_KEY)

# Define the strict output schema for the LLM
class MoAExtraction(TypedDict):
    moa_label: str  

model = genai.GenerativeModel(
    'models/gemini-2.5-flash-lite',
    generation_config={
        "response_mime_type": "application/json",
        "response_schema": MoAExtraction,
        "temperature": 0.0  # Low temperature for deterministic output
    }
)

# ==========================================
# 2. LLM Extraction Function (Robust Text Matching)
# ==========================================
def extract_iuphar_moa_llm(action, description, retries=7):
    """
    Calls the Gemini API to classify the MoA using Action and Assay Description.
    Returns plain text and strictly matches against valid labels without JSON parsing.
    """
    time.sleep(1) 
    
    prompt = f"""
    You are an expert pharmacologist.
    Determine the final Mechanism of Action (MoA) of a ligand based on its IUPHAR 'Action' label and 'Assay Description'.
    Keep in mind that 'Negative' action in an assay measuring IP1/Calcium mobilization often implies an 'antagonist', not necessarily a 'nam'.
    
    Classify into exactly one of the following labels:
    - "agonist" 
    - "antagonist"
    - "partial agonist"
    - "inverse agonist"
    - "pam" 
    - "nam" 
    - "binder" 
    
    Return ONLY the matching label in lowercase. Do not use quotes, code blocks, or extra text.
    
    Action: "{action}"
    Assay Description: "{description}"
    """
    
    valid_labels = ["inverse agonist", "partial agonist", "agonist", "antagonist", "pam", "nam", "binder"]
    wait = 5
    
    for attempt in range(retries):
        try:
            response = model.generate_content(prompt)
            raw_text = response.text.lower().strip()
            
            # Safely check if any of the valid labels exist in the raw response
            # Checked in specific order (e.g., 'inverse agonist' before 'agonist')
            for valid_label in valid_labels:
                if valid_label in raw_text:
                    return valid_label
                    
            # Fallback if the LLM produces unexpected output
            return "binder"
            
        except Exception as e:
            if "429" in str(e) or "ResourceExhausted" in str(e):
                print(f"Rate limit. Waiting {wait}s... (attempt {attempt+1}/{retries})")
                time.sleep(wait)
                wait *= 2
            else:
                print(f"API Error: {e}")
                break
                
    return "binder"

# ==========================================
# 3. Hybrid Processing Pipeline
# ==========================================
def process_iuphar_moa_hybrid(csv_path, output_dir):
    print("\n--- Processing IUPHAR MoA (Hybrid Rule + LLM) ---")
    
    df = pd.read_csv(csv_path)
    
    # Condition Separation: Rule-based vs LLM-based
    def determine_path(row):
        action = str(row['Action']).lower().strip()
        desc = str(row['Assay Description']).strip()
        has_desc = desc != 'nan' and desc != ''
        
        clear_actions = ['agonist', 'antagonist', 'partial agonist', 'inverse agonist']
        
        # Exact match for clear action with no description
        if not has_desc and action in clear_actions:
            return 'Rule'
        return 'LLM'
        
    df['Processing_Path'] = df.apply(determine_path, axis=1)
    
    # Apply Rule-based Mapping
    def apply_rule(action):
        action = str(action).lower().strip()
        if 'inverse agonist' in action: return 'inverse agonist'
        if 'partial agonist' in action: return 'partial agonist'
        if 'agonist' in action: return 'agonist'
        if 'antagonist' in action: return 'antagonist'
        return 'binder'
        
    rule_mask = df['Processing_Path'] == 'Rule'
    df.loc[rule_mask, 'MoA_Label'] = df.loc[rule_mask, 'Action'].apply(apply_rule)
    df.loc[rule_mask, 'Label_Source'] = 'Rule'
    
    # Apply LLM-based Mapping
    llm_mask = df['Processing_Path'] == 'LLM'
    llm_target_df = df[llm_mask].copy()
    
    # Extract unique combinations
    unique_pairs = llm_target_df[['Action', 'Assay Description']].drop_duplicates()
    unique_pairs['Action'] = unique_pairs['Action'].fillna('')
    unique_pairs['Assay Description'] = unique_pairs['Assay Description'].fillna('')
    
    print(f"Total rows: {len(df)} | Rule-based: {rule_mask.sum()} | LLM targets: {len(unique_pairs)}")
    
    CHECKPOINT_FILE = os.path.join(output_dir, "iuphar_moa_checkpoint.json")
    mapping_results = {}
    
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            mapping_results = json.load(f)
        print(f"Resumed from checkpoint: {len(mapping_results)} pairs already done.")

    # Process unique pairs via LLM
    for i, row in tqdm(unique_pairs.iterrows(), total=len(unique_pairs), desc='Mining MoA via LLM'):
        action_val = str(row['Action'])
        desc_val = str(row['Assay Description'])
        pair_key = f"{action_val} ||| {desc_val}"
        
        if pair_key in mapping_results:
            continue
            
        label = extract_iuphar_moa_llm(action_val, desc_val)
        mapping_results[pair_key] = label
        
        # Checkpoint save
        if (i + 1) % 50 == 0:
            with open(CHECKPOINT_FILE, "w") as f:
                json.dump(mapping_results, f, ensure_ascii=False)

    # Final checkpoint save
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(mapping_results, f, ensure_ascii=False)
        
    # Map results back to dataframe
    def map_llm_result(row):
        action_val = str(row['Action']).replace('nan', '')
        desc_val = str(row['Assay Description']).replace('nan', '')
        pair_key = f"{action_val} ||| {desc_val}"
        return mapping_results.get(pair_key, 'binder')

    df.loc[llm_mask, 'MoA_Label'] = df[llm_mask].apply(map_llm_result, axis=1)
    df.loc[llm_mask, 'Label_Source'] = 'LLM'
    
    # Final Cleanup
    df.drop(columns=['Processing_Path'], inplace=True)
    df.rename(columns={'Target UniProt ID': 'UniProt_AC'}, inplace=True)
    
    # Save Final Data
    out_path = os.path.join(output_dir, 'IUPHAR_MoA_Labeled.csv')
    df.to_csv(out_path, index=False)
    
    print("\n=== Final IUPHAR MoA Distribution ===")
    print(df['MoA_Label'].value_counts())
    print(f"\nSaved standardized IUPHAR MoA to: {out_path}")
    
    return df

In [ ]:
input_csv = str(BASE / "Output/DB/IUPHAR/NAR/IUPHAR_GPCR_Bioassay_Interaction.csv")
output_dir = str(BASE / "Output/DB/IUPHAR/NAR/")
os.makedirs(output_dir, exist_ok=True)

process_iuphar_moa_hybrid(input_csv, output_dir)

In [ ]:
"""
Fix Rule-based MoA labeling bug and regenerate IUPHAR_MoA_Labeled.csv
======================================================================
Bug: 'agonist' was checked before 'antagonist', so all antagonists
     were incorrectly labeled as 'agonist' in the Rule path.
Fix: Check longer/more specific strings first.
"""

import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from pathlib import Path

IUPHAR_PATH = Path(str(BASE / "Output/DB/IUPHAR/NAR/IUPHAR_GPCR_Bioassay_Interaction.csv"))
OUTPUT_PATH = Path(str(BASE / "Output/DB/IUPHAR/NAR/IUPHAR_MoA_Labeled.csv"))
CHECKPOINT  = Path(str(BASE / "Output/DB/IUPHAR/NAR/iuphar_moa_checkpoint.json"))

# ── 1. Fixed rule-based labeling (specific → general order) ──────────────────
def apply_rule_fixed(action: str) -> str:
    """
    Check longer/more specific strings first to avoid substring collision.
    'antagonist' must come before 'agonist'.
    'inverse agonist' and 'partial agonist' must come before plain 'agonist'.
    """
    a = str(action).lower().strip()
    if 'inverse agonist'  in a: return 'inverse agonist'
    if 'partial agonist'  in a: return 'partial agonist'
    if 'antagonist'       in a: return 'antagonist'   # ← moved before 'agonist'
    if 'agonist'          in a: return 'agonist'
    if 'pam'              in a: return 'pam'
    if 'nam'              in a: return 'nam'
    return 'binder'

# ── 2. Load existing labeled file and re-apply fix to Rule rows only ──────────
df = pd.read_csv(OUTPUT_PATH)

print("=== Before fix (Rule path only) ===")
rule_mask = df['Label_Source'] == 'Rule'
print(df[rule_mask]['MoA_Label'].value_counts())
print(f"\nSample antagonists mislabeled before fix:")
print(df[rule_mask & (df['Action'].str.lower().str.contains('antagonist', na=False))][
    ['Ligand','Action','MoA_Label']].head(10).to_string())

# Apply fix
df.loc[rule_mask, 'MoA_Label'] = df.loc[rule_mask, 'Action'].apply(apply_rule_fixed)

print("\n=== After fix (Rule path only) ===")
print(df[rule_mask]['MoA_Label'].value_counts())

print("\n=== Full distribution after fix ===")
print(df['MoA_Label'].value_counts())

# Save
df.to_csv(OUTPUT_PATH, index=False)
print(f"\nOverwritten: {OUTPUT_PATH}")


# ── 3. Quick sanity check: Action vs MoA_Label for Rule rows ─────────────────
print("\n=== Sanity check: Action → MoA_Label mapping (Rule path, n=20 sample) ===")
print(df[rule_mask][['Action','MoA_Label']].drop_duplicates().head(20).to_string())

In [ ]:
"""
Recompute IUPHAR vs Tier 1 BACC after the bug fix
"""
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from pathlib import Path

IUPHAR_LABELED = Path(str(BASE / "Output/DB/IUPHAR/NAR/IUPHAR_MoA_Labeled.csv"))
HUMAN_MOA_PATH = Path("./Output/DB/GPCRactDB/Human_MoA_Master_Integrated.csv")

MOA_MAP = {
    'agonist':         'Agonist',
    'partial agonist': 'Agonist',
    'antagonist':      'Antagonist',
    'inverse agonist': 'Antagonist',
}

iuphar = pd.read_csv(IUPHAR_LABELED)
iuphar['MoA_mapped'] = iuphar['MoA_Label'].map(MOA_MAP)
iuphar = iuphar[iuphar['MoA_mapped'].notna()].rename(
    columns={'InChIKey': 'Ligand_InChIKey', 'UniProt_AC': 'GPCR_UniProt'}
)

human_moa = pd.read_csv(HUMAN_MOA_PATH)
t1 = human_moa[human_moa['Final_Tier'] == 'Tier 1'].copy()
t1['Human_MoA_2'] = t1['Human_MoA'].map({
    'Agonist': 'Agonist', 'Partial Agonist': 'Agonist',
    'Antagonist': 'Antagonist', 'Inverse Agonist': 'Antagonist'
})
t1 = t1[t1['Human_MoA_2'].notna()]

merged = iuphar.merge(
    t1[['Ligand_InChIKey', 'GPCR_UniProt', 'Human_MoA_2']],
    on=['Ligand_InChIKey', 'GPCR_UniProt'], how='inner'
)

print(f"Matched pairs: {len(merged)}")
print(merged[['MoA_mapped','Human_MoA_2']].value_counts().sort_index())

if len(merged) >= 10:
    bacc = balanced_accuracy_score(merged['Human_MoA_2'], merged['MoA_mapped'])
    print(f"\nIUPHAR vs Tier 1 BACC (after bug fix): {bacc:.4f}")
else:
    print("Too few matched pairs — check join keys.")